# 转生千金 · Fish Audio Drama 3 日语配音一键批产与质检流水线（Colab 旗舰版）

本 Notebook 实现 **0 写死卡型、开箱即用、断点续传、四层工业级自动质检** 的全角色日语配音生产闭环：

| Colab GPU | 自动选型 | 显存策略 | 30 秒试音 (~50 句) | 主线+战斗 (~965 句) |
|---|---|---|---|---|
| **H100 80GB (`sm_90`)** | `fishaudio/drama-3` (FP8, 41.5 GB) / 可一键切 `Aratako/Drama-3-Japanese` | 原生 FP8 满血 | ~30 秒 | ~8–12 分钟 |
| **A100 80GB (`sm_80`)** | `fishaudio/drama-3-int4` (INT4, 22.1 GB, GPTQ-Marlin SM80 原生) | 避开 FP8 反量化爆显存 | ~40 秒 | ~12–16 分钟 |
| **A100 40GB (`sm_80`)** | `fishaudio/drama-3-int4` + 显存收紧参数 (`0.58 / 0.12 / 1024 / 3GiB / 20s`) | 单卡 40GB 稳定运行 | ~50 秒 | ~16–22 分钟 |
| **L4 (24GB) / T4 (16GB) / CPU** | **自动无缝切换 Fish Audio Cloud API** (`s2.1-pro-free` **$0 免费** 或 `drama-3`) | 直传 `reference_id`，无需拉参考音 | ~40 秒 | 视网络与配额 |

> **安全与工程纪律**：
> - `FISH_API_KEY` 仅从 Colab Secrets (`userdata`) 或运行时密码框读取，**绝不写入代码或仓库**。
> - 本地 SGLang-Omni 严格执行采样铁律：`temperature=1.0, top_p=0.9`，**严禁 `top_k` 与 `seed`**，动态计算 `max_new_tokens ∈ [220, 1024]`。
> - 全部成品自动静音修剪（留首 40ms / 尾 120ms）并双遍 `loudnorm` 至 `-16 LUFS / -1.5 dBTP / LRA=7`（`44.1 kHz mono MP3 64 kbps`）。

## Cell 1 · 自动探测 Colab GPU 规格 + 挂载 Google Drive 持久化缓存

自动检测当前 GPU 型号、显存与 Compute Capability (`sm_XX`)，生成最优部署配置，并将 HuggingFace 权重与成品语音缓存到 Google Drive，避免 Colab 断线重来。

In [ ]:
#@title 1. 检测 GPU 并挂载 Google Drive（自动适配 H100 / A100-80G / A100-40G / L4·T4 API 模式）
H100_MODEL_CHOICE = "fishaudio/drama-3" #@param ["fishaudio/drama-3", "Aratako/Drama-3-Japanese"]
CLOUD_API_MODEL = "s2.1-pro-free" #@param ["s2.1-pro-free", "drama-3"]
FORCE_CLOUD_API = False #@param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_WORK_DIR = "/content/drive/MyDrive/fictional_adventure_voice" #@param {type:"string"}

import json
import os
import subprocess
from pathlib import Path

def detect_gpu():
    # 优先使用 nvidia-smi，避免尚未安装 torch 时探测失败
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,compute_cap",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        if out:
            line = out.splitlines()[0]
            name, mem_mb_s, cc_s = [x.strip() for x in line.split(",")]
            mem_gb = float(mem_mb_s) / 1024.0
            major, minor = [int(x) for x in cc_s.split(".")[:2]]
            return {"has_gpu": True, "name": name, "vram_gb": round(mem_gb, 1), "sm": major * 10 + minor, "cc": cc_s}
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            mem_gb = props.total_memory / (1024 ** 3)
            sm = props.major * 10 + props.minor
            return {
                "has_gpu": True,
                "name": props.name,
                "vram_gb": round(mem_gb, 1),
                "sm": sm,
                "cc": f"{props.major}.{props.minor}",
            }
    except Exception:
        pass
    return {"has_gpu": False, "name": "None (CPU)", "vram_gb": 0.0, "sm": 0, "cc": "0.0"}

gpu = detect_gpu()
print(f"检测到硬件: {gpu['name']} | VRAM: {gpu['vram_gb']} GB | Compute Capability: sm_{gpu['sm']} ({gpu['cc']})")

# 严格四档自适应策略（绝不写死卡型）
if FORCE_CLOUD_API:
    tier_name = "Forced Cloud API"
    backend = "fish_api"
    model_id = CLOUD_API_MODEL
    extra_serve_args = []
    max_ref_seconds = 55.0
    max_ref_clips = 2
    default_concurrency = 2
elif gpu["has_gpu"] and gpu["sm"] >= 90 and gpu["vram_gb"] >= 70.0:
    tier_name = "H100/H200 80GB+ (SM90 原生 FP8)"
    backend = "sglang"
    model_id = H100_MODEL_CHOICE
    extra_serve_args = []
    max_ref_seconds = 55.0
    max_ref_clips = 2
    default_concurrency = 2
elif gpu["has_gpu"] and gpu["sm"] >= 80 and gpu["vram_gb"] >= 70.0:
    tier_name = "A100 80GB (SM80 INT4 GPTQ-Marlin 原生，规避 FP8 反量化爆显存)"
    backend = "sglang"
    model_id = "fishaudio/drama-3-int4"
    extra_serve_args = []
    max_ref_seconds = 55.0
    max_ref_clips = 2
    default_concurrency = 2
elif gpu["has_gpu"] and gpu["sm"] >= 80 and gpu["vram_gb"] >= 36.0:
    tier_name = "A100 40GB (SM80 INT4 + 显存收紧参数)"
    backend = "sglang"
    model_id = "fishaudio/drama-3-int4"
    extra_serve_args = [
        "--tts_engine.gpu_memory_fraction", "0.58",
        "--vocoder.gpu_memory_fraction", "0.12",
        "--tts_engine.factory.max_new_tokens", "1024",
        "--preprocessing.total_reserve_bytes", "3GiB",
        "--preprocessing.factory.max_reference_audio_seconds", "20",
    ]
    max_ref_seconds = 19.0
    max_ref_clips = 1
    default_concurrency = 1
else:
    tier_name = "L4 (24GB) / T4 (16GB) / 无大卡 -> 自动回退 Fish Audio Cloud API"
    backend = "fish_api"
    model_id = CLOUD_API_MODEL
    extra_serve_args = []
    max_ref_seconds = 55.0
    max_ref_clips = 2
    default_concurrency = 2
    print("\n" + "=" * 72)
    print("⚠️ 友好提示：本地运行 Fish Audio Drama 3 (35B MoE) 至少需要 >=40GB 显存且 SM80+ (A100/H100)。")
    print(f"   当前显卡为 {gpu['name']} ({gpu['vram_gb']} GB)，直接加载本地权重会触发 OOM！")
    print(f"   ✅ 已自动无缝切换至【Fish Audio Cloud API 模式】（当前模型：{model_id}）。")
    print("   💡 提示：`s2.1-pro-free` 为 $0 免费高配通道；`drama-3` 为付费旗舰演出模型。")
    print("      Cloud API 直接传递 `reference_id`，无需提前拉取参考音与下载 20-40GB 权重！")
    print("=" * 72 + "\n")

# 挂载 Google Drive 用于持久化 HF 缓存、断点续传进度与成品归档
drive_ready = False
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        Path(DRIVE_WORK_DIR).mkdir(parents=True, exist_ok=True)
        (Path(DRIVE_WORK_DIR) / "hf_cache").mkdir(parents=True, exist_ok=True)
        (Path(DRIVE_WORK_DIR) / "custom_refs").mkdir(parents=True, exist_ok=True)
        (Path(DRIVE_WORK_DIR) / "checkpoint_voice").mkdir(parents=True, exist_ok=True)
        os.environ["HF_HOME"] = str(Path(DRIVE_WORK_DIR) / "hf_cache")
        os.environ["HUGGINGFACE_HUB_CACHE"] = str(Path(DRIVE_WORK_DIR) / "hf_cache" / "hub")
        drive_ready = True
        print(f"✅ Google Drive 已挂载，持久化目录: {DRIVE_WORK_DIR}")
    except Exception as e:
        print(f"⚠️ 未能挂载 Google Drive（若非 Colab 环境可忽略）: {e}")

cfg = {
    "gpu": gpu,
    "tier_name": tier_name,
    "backend": backend,
    "model_id": model_id,
    "cloud_api_model": CLOUD_API_MODEL,
    "extra_serve_args": extra_serve_args,
    "max_ref_seconds": max_ref_seconds,
    "max_ref_clips": max_ref_clips,
    "default_concurrency": default_concurrency,
    "drive_ready": drive_ready,
    "drive_work_dir": DRIVE_WORK_DIR if drive_ready else "",
    "repo_dir": "/content/fictional-adventure",
}
Path("/content/voice_runtime_cfg.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n📋 本次运行自动选型摘要：")
print(f"  策略档位       : {tier_name}")
print(f"  推理后端       : {backend}")
print(f"  使用模型       : {model_id}")
print(f"  额外启动参数   : {' '.join(extra_serve_args) if extra_serve_args else '(默认)'}")
print(f"  参考音时长上限 : {max_ref_seconds}s (单声线最多 {max_ref_clips} 段)")


## Cell 2 · 克隆/同步游戏仓库 & 按模式安装依赖

- **本地 `sglang` 模式**：安装 `ffmpeg`、`uv`、`sglang-omni[fishaudio]`、`huggingface_hub`；
- **云端 `fish_api` 模式**：仅装 `ffmpeg` + `requests`（~10 秒极速完成！）；
- **可选 `ENABLE_ASR_QC`**：安装 `transformers`、`accelerate`、`pykakasi`、`soundfile` 以启用 `litagin/anime-whisper` 第 3 层反向转写质检。

In [ ]:
#@title 2. 同步仓库并安装 FFmpeg / SGLang-Omni / 可选 Anime-Whisper 质检依赖
REPO_URL = "https://github.com/juanhanyu/fictional-adventure.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
ENABLE_ASR_QC = True #@param {type:"boolean"}

import json
import os
import shutil
import subprocess
from pathlib import Path

cfg = json.loads(Path("/content/voice_runtime_cfg.json").read_text(encoding="utf-8"))
cfg["enable_asr_qc"] = bool(ENABLE_ASR_QC)
repo_dir = Path(cfg["repo_dir"])

def run_cmd(cmd, cwd=None, check=True):
    print(f"$ {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd, cwd=cwd)
    if check and proc.returncode != 0:
        raise RuntimeError(f"命令失败 (exit={proc.returncode}): {' '.join(cmd)}")
    return proc.returncode

# 1. 克隆或同步仓库
if not (repo_dir / ".git").exists():
    run_cmd(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
else:
    run_cmd(["git", "fetch", "origin", REPO_BRANCH, "--depth", "1"], cwd=str(repo_dir))
    run_cmd(["git", "checkout", REPO_BRANCH], cwd=str(repo_dir))
    run_cmd(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=str(repo_dir))

# 2. 确保 ffmpeg 与 uv 已安装
if not shutil.which("ffmpeg"):
    run_cmd(["apt-get", "-qq", "update"])
    run_cmd(["apt-get", "-qq", "-y", "install", "ffmpeg"])
else:
    print("✅ ffmpeg 已就绪")

run_cmd(["pip", "install", "-q", "-U", "uv", "requests", "huggingface_hub"])

# 3. 按后端安装 TTS 依赖
if cfg["backend"] == "sglang":
    print("🚀 正在安装 sglang-omni[fishaudio]（首次安装约需 1-3 分钟）...", flush=True)
    rc = run_cmd(
        ["uv", "pip", "install", "--system", "--prerelease=allow", "sglang-omni[fishaudio]"],
        check=False,
    )
    if rc != 0:
        print("⚠️ PyPI extra 安装失败，回退至官方 GitHub 源码安装 sglang-omni[fishaudio]...", flush=True)
        sgl_src = Path("/content/sglang-omni")
        if not sgl_src.exists():
            run_cmd(["git", "clone", "--depth", "1", "https://github.com/sgl-project/sglang-omni.git", str(sgl_src)])
        run_cmd(["uv", "pip", "install", "--system", "--prerelease=allow", "-e", ".[fishaudio]"], cwd=str(sgl_src))
else:
    print("✅ 当前为 Fish Audio Cloud API 轻量模式，无需安装 SGLang-Omni。")

# 4. 可选：安装第 3 层 ASR 反向转写质检依赖（litagin/anime-whisper + pykakasi）
if ENABLE_ASR_QC:
    print("🔍 正在安装 litagin/anime-whisper 质检依赖 (transformers / accelerate / pykakasi)...", flush=True)
    run_cmd(["uv", "pip", "install", "--system", "transformers", "accelerate", "pykakasi", "soundfile"])

# 5. 检查 Node.js（用于自动构建 js/voice-manifest.js）
if shutil.which("node"):
    node_ver = subprocess.check_output(["node", "--version"], text=True).strip()
    print(f"✅ Node.js 已就绪: {node_ver}")
else:
    print("⚠️ 未检测到 Node.js，稍后生成 voice-manifest.js 时可在本地仓库补跑 `node tools/voice/build_manifest.mjs`。")

Path("/content/voice_runtime_cfg.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n✅ Cell 2 环境准备完毕！")


## Cell 3 · 安全配置 `FISH_API_KEY` + 自动拉取 25 套角色参考音并试听

- **密钥安全**：优先读取 Colab 左侧 🔑 Secrets 中名为 `FISH_API_KEY` 的密钥；未配置时通过隐藏密码框输入，**绝不硬编码**。
- **自定义干声覆写**：若您在 Google Drive 的 `fictional_adventure_voice/custom_refs/` 中放入 `<profileId>.wav` + `.txt`，将自动同步覆盖对应角色参考音。
- **Cloud API 模式**：`fish_api` 模式下直接传 `reference_id`，可跳过拉取参考音；若想先试听 25 位原声模型，勾选 `FETCH_REFS_EVEN_IN_API_MODE` 即可（查元数据零 TTS 计费）。

In [ ]:
#@title 3. 配置 FISH_API_KEY、拉取 25 套参考音并生成内嵌试听控件
VERIFY_REF_ASR = False #@param {type:"boolean"}
FETCH_REFS_EVEN_IN_API_MODE = False #@param {type:"boolean"}
FORCE_REFETCH = False #@param {type:"boolean"}

import base64
import getpass
import html
import json
import os
import shutil
import subprocess
from pathlib import Path
from IPython.display import HTML, display

cfg = json.loads(Path("/content/voice_runtime_cfg.json").read_text(encoding="utf-8"))
repo_dir = Path(cfg["repo_dir"])

# 1. 安全读取 FISH_API_KEY（Colab Secrets -> 环境变量 -> getpass，绝不硬编码）
api_key = ""
try:
    from google.colab import userdata
    api_key = (userdata.get("FISH_API_KEY") or "").strip()
    if api_key:
        print("✅ 已从 Colab Secrets 读取 FISH_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("FISH_API_KEY", "").strip()
if not api_key:
    api_key = getpass.getpass("请输入 FISH_API_KEY（输入隐藏，仅存当前会话内存）: ").strip()

if api_key:
    os.environ["FISH_API_KEY"] = api_key
else:
    print("⚠️ 未输入 FISH_API_KEY；若仅使用本地 custom_refs 可继续，访问 Fish Audio API 将受限。")

# 2. 同步 Google Drive 中的自定义干声覆写（custom_refs/<profileId>.wav + .txt）
custom_local = repo_dir / "tools/voice/custom_refs"
custom_local.mkdir(parents=True, exist_ok=True)
if cfg.get("drive_ready") and cfg.get("drive_work_dir"):
    drive_custom = Path(cfg["drive_work_dir"]) / "custom_refs"
    if drive_custom.is_dir():
        copied = 0
        for p in drive_custom.iterdir():
            if p.is_file():
                shutil.copy2(p, custom_local / p.name)
                copied += 1
        if copied:
            print(f"✅ 已从 Google Drive 同步 {copied} 个 custom_refs 本地干声文件")

# 3. 执行 fetch_fish_refs.py
refs_dir = repo_dir / "tools/voice/refs_cache"
refs_resolved_path = repo_dir / "tools/voice/refs_resolved.json"
need_fetch = (cfg["backend"] == "sglang") or FETCH_REFS_EVEN_IN_API_MODE

if need_fetch:
    cmd = [
        "python3",
        "tools/voice/fetch_fish_refs.py",
        "--refs-dir", str(refs_dir),
        "--custom-refs-dir", str(custom_local),
        "--out", str(refs_resolved_path),
        "--max-seconds", str(cfg["max_ref_seconds"]),
        "--max-clips", str(cfg["max_ref_clips"]),
    ]
    if VERIFY_REF_ASR:
        cmd.append("--verify-asr")
    if FORCE_REFETCH:
        cmd.append("--force")

    print(f"$ {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd, cwd=str(repo_dir))
    if proc.returncode != 0:
        print("⚠️ 部分声线参考音拉取存在失败，请检查上方日志及下方列表。")

    # 渲染 Notebook 内置 HTML 音频试听墙（base64 内嵌，Colab 100% 可播）
    if refs_resolved_path.is_file():
        rdata = json.loads(refs_resolved_path.read_text(encoding="utf-8"))
        rows = []
        for pid, pinfo in rdata.get("profiles", {}).items():
            source = pinfo.get("source", "unknown")
            title = pinfo.get("fishModelTitle") or pinfo.get("fishModelId") or "-"
            badge_bg = "#dcfce7" if source in ("primary", "custom_override") else ("#fef3c7" if source == "backup" else "#fee2e2")
            clips_html = []
            for ref in pinfo.get("references", []):
                ap = Path(ref["audio_path"])
                if not ap.is_file():
                    continue
                b64 = base64.b64encode(ap.read_bytes()).decode("ascii")
                clips_html.append(
                    f"<div style='margin:4px 0;'>"
                    f"<audio controls preload='none' style='height:30px;width:270px;' src='data:audio/wav;base64,{b64}'></audio>"
                    f"<div style='font-size:12px;color:#334155;'>({ref['duration_sec']:.1f}s · {html.escape(str(ref.get('text_source','metadata')))}) "
                    f"{html.escape(ref.get('text',''))}</div></div>"
                )
            rows.append(
                f"<tr><td style='padding:8px;border:1px solid #e2e8f0;'><b>{html.escape(pid)}</b><br>"
                f"<span style='font-size:12px;color:#64748b;'>{html.escape(str(pinfo.get('displayName','')))}</span></td>"
                f"<td style='padding:8px;border:1px solid #e2e8f0;'><span style='background:{badge_bg};padding:2px 6px;border-radius:3px;font-size:12px;'>"
                f"{html.escape(source)}</span><div style='font-size:11px;color:#475569;margin-top:4px;'>{html.escape(str(title))}</div></td>"
                f"<td style='padding:8px;border:1px solid #e2e8f0;'>{''.join(clips_html) or '❌ 缺失'}</td></tr>"
            )
        summary = rdata.get("summary", {})
        display(HTML(
            f"<div style='font-family:system-ui,sans-serif;background:#f8fafc;padding:12px;border:1px solid #cbd5e1;'>"
            f"<h3 style='margin:0 0 8px 0;color:#1e3a8a;'>🎙️ 角色参考音试听表（成功 {summary.get('resolved',0)}/{summary.get('totalProfiles',0)}）</h3>"
            f"<table style='width:100%;border-collapse:collapse;background:#fff;'>"
            f"<thead><tr style='background:#eff6ff;'><th style='padding:8px;border:1px solid #cbd5e1;'>Profile</th>"
            f"<th style='padding:8px;border:1px solid #cbd5e1;'>来源/模型</th>"
            f"<th style='padding:8px;border:1px solid #cbd5e1;'>44.1kHz 参考样本与日文文本</th></tr></thead>"
            f"<tbody>{''.join(rows)}</tbody></table></div>"
        ))
else:
    print("✅ Cloud API 模式下将直接使用 casting_fish_audio.json 中的 reference_id，已跳过参考音下载。")


## Cell 4 · 后台启动 `sgl-omni serve` 并等待 `/health` 就绪

- 自动按 Cell 1 探测的显卡档位传入最优模型路径、显存参数与 `--allowed-local-media-path <REF_DIR>`；
- 首次启动会下载权重并捕获 CUDA Graph（约需 2–6 分钟），Cell 会实时输出 `/content/sgl_omni_serve.log` 末尾日志直至 `/health` 返回 HTTP 200；
- 若当前处于 **Fish Audio Cloud API 模式**，本单元格会自动跳过本地服务启动。

In [ ]:
#@title 4. 后台启动 sgl-omni serve 并轮询等待 /health 就绪
SERVE_PORT = 8000 #@param {type:"integer"}
STARTUP_TIMEOUT_SEC = 900 #@param {type:"integer"}

import json
import os
import subprocess
import time
import urllib.request
from pathlib import Path

cfg = json.loads(Path("/content/voice_runtime_cfg.json").read_text(encoding="utf-8"))
repo_dir = Path(cfg["repo_dir"])
log_path = Path("/content/sgl_omni_serve.log")

def check_health(port: int) -> bool:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3) as resp:
            return getattr(resp, "status", 200) == 200
    except Exception:
        return False

if cfg["backend"] != "sglang":
    print(f"ℹ️ 当前为 Fish Audio Cloud API 模式 (model={cfg['model_id']})，无需启动本地 sgl-omni serve。")
else:
    if check_health(SERVE_PORT):
        print(f"✅ 检测到本地 sgl-omni serve (:{SERVE_PORT}) 已在线且 /health 正常，直接复用！")
    else:
        refs_abs = str((repo_dir / "tools/voice/refs_cache").resolve())
        Path(refs_abs).mkdir(parents=True, exist_ok=True)
        model_id = cfg["model_id"]

        # 1. 预下载模型（复用 HF_HOME / Google Drive 缓存）
        print(f"📦 正在核对/下载模型权重: {model_id} ...", flush=True)
        dl_cmd = ["huggingface-cli", "download", model_id]
        subprocess.run(dl_cmd, check=False)

        # 2. 启动 sgl-omni serve 后台服务
        serve_cmd = [
            "sgl-omni", "serve",
            "--model-path", model_id,
            "--port", str(SERVE_PORT),
            "--allowed-local-media-path", refs_abs,
        ] + list(cfg.get("extra_serve_args") or [])

        print(f"🚀 后台启动指令:\n   {' '.join(serve_cmd)}", flush=True)
        log_fp = log_path.open("w", encoding="utf-8")
        proc = subprocess.Popen(
            serve_cmd,
            stdout=log_fp,
            stderr=subprocess.STDOUT,
            cwd=str(repo_dir),
            env=os.environ.copy(),
        )
        cfg["sglang_pid"] = proc.pid
        cfg["sglang_port"] = SERVE_PORT
        Path("/content/voice_runtime_cfg.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")

        # 3. 轮询 /health 并输出实时日志尾部
        t0 = time.monotonic()
        last_tail = ""
        while time.monotonic() - t0 < STARTUP_TIMEOUT_SEC:
            if check_health(SERVE_PORT):
                elapsed = int(time.monotonic() - t0)
                print(f"\n✅ sgl-omni serve 已就绪！(耗时 {elapsed}s, PID={proc.pid}, /health HTTP 200)")
                break
            if proc.poll() is not None:
                tail = log_path.read_text(encoding="utf-8", errors="replace")[-3000:]
                raise RuntimeError(f"❌ sgl-omni serve 提前退出 (code={proc.returncode})！日志末尾：\n{tail}")
            try:
                lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
                tail_line = lines[-1] if lines else "(等待输出...)"
                if tail_line != last_tail:
                    print(f"  [{int(time.monotonic() - t0):3d}s] {tail_line[:140]}", flush=True)
                    last_tail = tail_line
            except Exception:
                pass
            time.sleep(5)
        else:
            tail = log_path.read_text(encoding="utf-8", errors="replace")[-2000:]
            raise TimeoutError(f"等待 /health 超时 ({STARTUP_TIMEOUT_SEC}s)。请检查 {log_path}：\n{tail}")


## Cell 5 · 30 秒试音模式（`--scope audition`）+ 内置 25 角色定妆试听墙

在正式全量批产前，先花 **30–60 秒** 为全部 **25 位角色各合成 2 句代表性台词（~50 句）**，直接在 Notebook 内试听定妆音色与情绪表现！
如发现某位角色音色不贴合，可立即去 `casting_fish_audio.json` 更换模型或上传自定义干声到 `custom_refs/`，再单角色重跑。

In [ ]:
#@title 5. 运行 30 秒试音包 (--scope audition) 并渲染全角色 HTML 音频试听器
AUDITION_ASR_QC = True #@param {type:"boolean"}
AUDITION_FORCE = False #@param {type:"boolean"}

import base64
import html
import json
import subprocess
from collections import defaultdict
from pathlib import Path
from IPython.display import HTML, display

cfg = json.loads(Path("/content/voice_runtime_cfg.json").read_text(encoding="utf-8"))
repo_dir = Path(cfg["repo_dir"])
port = int(cfg.get("sglang_port", 8000))

cmd = [
    "python3",
    "tools/voice/batch_synthesize.py",
    "--backend", cfg["backend"],
    "--scope", "audition",
    "--model-id", cfg["model_id"],
    "--max-ref-clips", str(cfg["max_ref_clips"]),
    "--concurrency", str(cfg["default_concurrency"]),
    "--max-retries", "3",
]
if cfg["backend"] == "sglang":
    cmd += ["--sglang-url", f"http://127.0.0.1:{port}/v1/audio/speech"]
else:
    cmd += ["--fish-api-model", cfg["cloud_api_model"]]
if AUDITION_ASR_QC and cfg.get("enable_asr_qc", False):
    cmd.append("--asr-qc")
if AUDITION_FORCE:
    cmd.append("--force")

print(f"$ {' '.join(cmd)}", flush=True)
subprocess.run(cmd, cwd=str(repo_dir), check=False)

# 读取缓存与台本，构建按角色分组的内嵌 HTML 试听器
cache_path = repo_dir / "tools/voice/.synth_cache.json"
script_path = repo_dir / "tools/voice/script_jp_master.json"
out_dir = repo_dir / "assets/voice"

if cache_path.is_file() and script_path.is_file():
    cache = json.loads(cache_path.read_text(encoding="utf-8"))
    sraw = json.loads(script_path.read_text(encoding="utf-8"))
    lines = sraw if isinstance(sraw, list) else (sraw.get("lines") or [])
    grouped = defaultdict(list)
    for ln in lines:
        vid = ln.get("voiceId") or ln.get("voice_id")
        if vid in cache:
            rec = cache[vid]
            mp3_path = out_dir / rec.get("audioRelPath", "")
            if mp3_path.is_file():
                grouped[rec.get("profileId", "unknown")].append((rec, mp3_path))

    blocks = []
    for pid in sorted(grouped.keys()):
        items = grouped[pid][:4]  # 试音视图每个声线展示前几句
        item_rows = []
        for rec, mp3_path in items:
            b64 = base64.b64encode(mp3_path.read_bytes()).decode("ascii")
            cer_txt = f"CER {rec['cer']*100:.1f}%" if rec.get("cer") is not None else "ASR关"
            status_color = "#15803d" if rec.get("qaPassed", True) else "#b45309"
            item_rows.append(
                f"<div style='padding:8px 0;border-top:1px solid #e2e8f0;'>"
                f"<div style='display:flex;align-items:center;gap:10px;flex-wrap:wrap;'>"
                f"<code style='font-size:12px;color:#1e40af;'>{html.escape(rec['voiceId'])}</code>"
                f"<audio controls preload='none' style='height:30px;width:260px;' src='data:audio/mpeg;base64,{b64}'></audio>"
                f"<span style='font-size:12px;color:{status_color};'>{rec.get('durationSec',0):.2f}s · {cer_txt} · try={rec.get('attempts',1)}</span>"
                f"</div>"
                f"<div style='font-size:13px;color:#0f172a;margin-top:4px;'><b>JP:</b> {html.escape(str(rec.get('jpText','')))}</div>"
                f"<div style='font-size:12px;color:#475569;'><b>ZH:</b> {html.escape(str(rec.get('zhText','')))}</div>"
                f"</div>"
            )
        blocks.append(
            f"<div style='background:#fff;border:1px solid #cbd5e1;margin:10px 0;padding:10px 14px;'>"
            f"<div style='font-weight:700;color:#0f172a;font-size:14px;'>🎭 {html.escape(pid)}（已生成 {len(grouped[pid])} 句）</div>"
            f"{''.join(item_rows)}</div>"
        )
    display(HTML(
        f"<div style='font-family:system-ui,sans-serif;background:#f8fafc;padding:12px;border:1px solid #93c5fd;'>"
        f"<h3 style='margin:0 0 6px 0;color:#1d4ed8;'>🎧 25 角色定妆试音墙（共 {len(grouped)} 位角色有试音成品）</h3>"
        f"<div style='font-size:12px;color:#475569;margin-bottom:8px;'>若某句不满意，在 Cell 6 填入 <code>ONLY_IDS</code> 即可单句重抽！</div>"
        f"{''.join(blocks)}</div>"
    ))
else:
    print("⚠️ 未检测到生成的试音成品，请确认 script_jp_master.json 存在且 Cell 5 无报错。")


## Cell 6 · 一键批量生产（`p1` / `main` / `all`）+ 四层质检 + 响度归一化 + 构建 `voice-manifest.js`

- **SHA-256 断点续传**：已通过试音/上次跑完的条目会自动秒跳过；若 Colab 中途断线，重连后从 Cell 1 跑到这里即可无缝续产！
- **单句精准返修**：在 `ONLY_IDS` 填入逗号分隔的 `voiceId`（如 `v_ch1_0012,v_combat_sylvia_03`）并勾选 `FORCE_REROLL`，即可精准重抽指定台词。

In [ ]:
#@title 6. 批量生成全量/分档语音 + 自动四层质检 + 同步 Google Drive 断点缓存
SCOPE = "main" #@param ["p1", "main", "all"]
ENABLE_ASR_QC_BATCH = True #@param {type:"boolean"}
MAX_RETRIES = 3 #@param {type:"integer"}
CONCURRENCY = 0 #@param {type:"integer"}
ONLY_IDS = "" #@param {type:"string"}
FORCE_REROLL = False #@param {type:"boolean"}

import json
import shutil
import subprocess
from pathlib import Path

cfg = json.loads(Path("/content/voice_runtime_cfg.json").read_text(encoding="utf-8"))
repo_dir = Path(cfg["repo_dir"])
port = int(cfg.get("sglang_port", 8000))
eff_concurrency = CONCURRENCY if CONCURRENCY > 0 else int(cfg["default_concurrency"])

# 1. 运行前：若 Google Drive 有上次断点续传缓存与已产 MP3，先恢复到本地工作区
drive_ckpt = Path(cfg["drive_work_dir"]) / "checkpoint_voice" if cfg.get("drive_ready") else None
if drive_ckpt and drive_ckpt.is_dir():
    d_cache = drive_ckpt / ".synth_cache.json"
    l_cache = repo_dir / "tools/voice/.synth_cache.json"
    if d_cache.is_file() and not l_cache.is_file():
        shutil.copy2(d_cache, l_cache)
        print("✅ 已从 Google Drive 恢复 SHA-256 断点缓存")
    d_assets = drive_ckpt / "assets_voice"
    l_assets = repo_dir / "assets/voice"
    if d_assets.is_dir() and not any(l_assets.rglob("*.mp3")) if l_assets.exists() else d_assets.is_dir():
        shutil.copytree(d_assets, l_assets, dirs_exist_ok=True)
        print("✅ 已从 Google Drive 恢复历史成品 MP3")

# 2. 执行批产 + 四层质检 + 双遍 loudnorm + 自动 build_manifest.mjs
cmd = [
    "python3",
    "tools/voice/batch_synthesize.py",
    "--backend", cfg["backend"],
    "--scope", SCOPE,
    "--model-id", cfg["model_id"],
    "--max-ref-clips", str(cfg["max_ref_clips"]),
    "--concurrency", str(eff_concurrency),
    "--max-retries", str(MAX_RETRIES),
]
if cfg["backend"] == "sglang":
    cmd += ["--sglang-url", f"http://127.0.0.1:{port}/v1/audio/speech"]
else:
    cmd += ["--fish-api-model", cfg["cloud_api_model"]]
if ENABLE_ASR_QC_BATCH and cfg.get("enable_asr_qc", False):
    cmd.append("--asr-qc")
if ONLY_IDS.strip():
    cmd += ["--only-ids", ONLY_IDS.strip()]
if FORCE_REROLL:
    cmd.append("--force")

print(f"$ {' '.join(cmd)}", flush=True)
proc = subprocess.run(cmd, cwd=str(repo_dir))

# 3. 运行后：增量备份 SHA-256 缓存与成品 MP3 到 Google Drive 防断线丢失
if drive_ckpt:
    drive_ckpt.mkdir(parents=True, exist_ok=True)
    l_cache = repo_dir / "tools/voice/.synth_cache.json"
    if l_cache.is_file():
        shutil.copy2(l_cache, drive_ckpt / ".synth_cache.json")
    l_assets = repo_dir / "assets/voice"
    if l_assets.is_dir():
        shutil.copytree(l_assets, drive_ckpt / "assets_voice", dirs_exist_ok=True)
    print(f"💾 已同步断点缓存与成品音频至 Google Drive: {drive_ckpt}")

if proc.returncode != 0:
    print("⚠️ 批产完成但存在少量失败条目，请在 Cell 7 查看 QA 报告与复听清单。")
else:
    print("\n🎉 Cell 6 批量生产与 manifest 构建全部完成！")


## Cell 7 · QA 质检汇总、高 CER 人工复听抽查 & 打包 `voice_release_pack.tar.gz` 至 Google Drive

- 展示 `tools/voice/qa_report.json` 核心指标，并为 **高 CER / 带警告句** 生成内嵌音频播放器供快速点听抽查；
- 将 `assets/voice/`、`js/voice-manifest.js`、`tools/voice/qa_report.json`、`tools/voice/refs_resolved.json` 打包为 `voice_release_pack.tar.gz` 存到 Google Drive；
- 附带**可选单次合入 Git 仓库指引**（推荐先复听确认后再单次提交，避免多次推送二进制音频撑爆 `.git` 历史体积）。

In [ ]:
#@title 7. 汇总 QA 报告、高 CER 复听抽查并打包 voice_release_pack.tar.gz 到 Google Drive
MAX_REVIEW_PLAYERS = 30 #@param {type:"integer"}

import base64
import hashlib
import html
import json
import tarfile
from pathlib import Path
from IPython.display import HTML, display

cfg = json.loads(Path("/content/voice_runtime_cfg.json").read_text(encoding="utf-8"))
repo_dir = Path(cfg["repo_dir"])
qa_path = repo_dir / "tools/voice/qa_report.json"

if not qa_path.is_file():
    raise FileNotFoundError(f"未找到 QA 报告: {qa_path}，请先运行 Cell 5 或 Cell 6。")

qa = json.loads(qa_path.read_text(encoding="utf-8"))
st = qa.get("stats", {})

print("=" * 68)
print("📊 Fish Audio Drama 3 批产质检汇总报告")
print("=" * 68)
print(f"  后端 / 模型     : {qa.get('backend')} / {qa.get('modelId')}")
print(f"  生产范围        : {qa.get('scope')}")
print(f"  总选中 / 成功   : {st.get('totalSelected')} / {st.get('succeeded')} (成功率 {st.get('successRatePct')}%)")
print(f"  缓存跳过 / 失败 : {st.get('cachedSkipped')} / {st.get('failed')}")
print(f"  自动重抽句数    : {st.get('retriedCount')}")
print(f"  待人工复听句数  : {st.get('flaggedForReviewCount')}")
print(f"  平均时长 / 总长 : {st.get('avgDurationSec')}s / {st.get('totalDurationSec')}s")
if st.get("avgCer") is not None:
    print(f"  平均 Kana CER   : {st.get('avgCer') * 100:.2f}%")
print(f"  语音包总体积    : {st.get('totalPackSizeMB')} MB (MP3 64kbps mono -16 LUFS)")
print(f"  Manifest 状态   : {qa.get('manifestBuild', {}).get('message')}")
print("=" * 68)

# 渲染高 CER / 告警复听抽查列表
review_list = qa.get("highCerReviewList") or []
if review_list:
    rows = []
    out_dir = repo_dir / "assets/voice"
    for item in review_list[:MAX_REVIEW_PLAYERS]:
        mp3_path = out_dir / item.get("audioRelPath", "")
        player_html = "<span>文件缺失</span>"
        if mp3_path.is_file():
            b64 = base64.b64encode(mp3_path.read_bytes()).decode("ascii")
            player_html = f"<audio controls preload='none' style='height:28px;width:220px;' src='data:audio/mpeg;base64,{b64}'></audio>"
        cer_s = f"{item['cer']*100:.1f}%" if item.get("cer") is not None else "-"
        issues_s = "; ".join(item.get("issues") or [])
        rows.append(
            f"<tr>"
            f"<td style='padding:6px;border:1px solid #e2e8f0;'><code>{html.escape(str(item['voiceId']))}</code><br>{html.escape(str(item['profileId']))}</td>"
            f"<td style='padding:6px;border:1px solid #e2e8f0;color:#b45309;font-weight:600;'>{cer_s}<div style='font-size:11px;font-weight:400;'>{html.escape(issues_s)}</div></td>"
            f"<td style='padding:6px;border:1px solid #e2e8f0;'>{player_html}</td>"
            f"<td style='padding:6px;border:1px solid #e2e8f0;font-size:12px;'>"
            f"<div><b>原台词:</b> {html.escape(str(item.get('jpText','')))}</div>"
            f"<div style='color:#475569;'><b>ASR识:</b> {html.escape(str(item.get('asrText') or '-'))}</div>"
            f"</td></tr>"
        )
    display(HTML(
        f"<div style='font-family:system-ui,sans-serif;background:#fffbeb;padding:12px;border:1px solid #fcd34d;margin-top:10px;'>"
        f"<h4 style='margin:0 0 8px 0;color:#92400e;'>🔎 需人工复听的高 CER / 警告句（展示前 {min(len(review_list), MAX_REVIEW_PLAYERS)} / 共 {len(review_list)} 句）</h4>"
        f"<table style='width:100%;border-collapse:collapse;background:#fff;'>"
        f"<thead><tr style='background:#fef3c7;'><th style='padding:6px;border:1px solid #fde68a;'>ID / 角色</th>"
        f"<th style='padding:6px;border:1px solid #fde68a;'>CER / 告警</th>"
        f"<th style='padding:6px;border:1px solid #fde68a;'>试听</th>"
        f"<th style='padding:6px;border:1px solid #fde68a;'>原台词 vs ASR 转写</th></tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table></div>"
    ))
else:
    print("✅ 零高 CER / 零质检警告句，全部一次通过！")

# 打包 voice_release_pack.tar.gz（仅含成品音频、manifest、QA 报告，不含临时缓存）
pack_target = (
    Path(cfg["drive_work_dir"]) / "voice_release_pack.tar.gz"
    if cfg.get("drive_ready") and cfg.get("drive_work_dir")
    else Path("/content/voice_release_pack.tar.gz")
)
pack_target.parent.mkdir(parents=True, exist_ok=True)

include_paths = [
    "assets/voice",
    "js/voice-manifest.js",
    "tools/voice/qa_report.json",
    "tools/voice/refs_resolved.json",
]
added = 0
with tarfile.open(pack_target, "w:gz") as tar:
    for rel in include_paths:
        full = repo_dir / rel
        if full.exists():
            tar.add(full, arcname=rel)
            added += 1

sha = hashlib.sha256(pack_target.read_bytes()).hexdigest()[:16]
pack_mb = pack_target.stat().st_size / (1024 * 1024)
print(f"\n📦 发布包已生成: {pack_target}")
print(f"   包含条目数: {added} | 压缩体积: {pack_mb:.2f} MB | SHA256(前16位): {sha}")
print(
    "\n💡 【合入本地仓库推荐做法（防多次推送二进制撑爆 .git）】\n"
    "   1. 从 Google Drive 下载 `voice_release_pack.tar.gz` 到本地仓库根目录；\n"
    "   2. 解压覆盖并重建索引：\n"
    "      tar -xzf voice_release_pack.tar.gz && node tools/voice/build_manifest.mjs\n"
    "   3. 本地抽听验收通过后，再执行单次正式提交：\n"
    "      git add assets/voice js/voice-manifest.js tools/voice/qa_report.json\n"
    "      git commit -m 'feat(voice): import Fish Audio Drama 3 JP voice pack' && git push origin main"
)
